In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# 1. Load dataset
df = pd.read_csv("fakenews.csv")

# 2. Handle missing values
df["headline"] = df["headline"].fillna("").astype(str)
df["body_text"] = df["body_text"].fillna("").astype(str)

# Combine headline and body text
df["text"] = df["headline"] + " " + df["body_text"]

# 3. Calculate Term Frequency (TF)
tf_vectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=1000
)

tf_matrix = tf_vectorizer.fit_transform(df["text"])
words = tf_vectorizer.get_feature_names_out()

# Normalize term frequencies by document length
document_lengths = tf_matrix.sum(axis=1).A1
tf_normalized = tf_matrix.astype(float).multiply(
    (1 / document_lengths.clip(min=1))[:, None]
)

# 4. Calculate Inverse Document Frequency (IDF)
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=1000
)

tfidf_matrix = tfidf_vectorizer.fit_transform(df["text"])
tfidf_words = tfidf_vectorizer.get_feature_names_out()

# IDF values
idf_values = tfidf_vectorizer.idf_

# Average TF-IDF for each word
average_tfidf = tfidf_matrix.mean(axis=0).A1

# Total word frequency across the dataset
total_tf = tf_matrix.sum(axis=0).A1

# 5. Create results DataFrame
tf_idf_df = pd.DataFrame({
    "Word": words,
    "Total_TF": total_tf,
    "IDF": idf_values,
    "Average_TF_IDF": average_tfidf
})

# 6. Display the most frequent words
print("Top 20 Words by Term Frequency:")
print(
    tf_idf_df.sort_values(
        by="Total_TF",
        ascending=False
    ).head(20)
)

# Display words with highest average TF-IDF
print("\nTop 20 Words by Average TF-IDF:")
print(
    tf_idf_df.sort_values(
        by="Average_TF_IDF",
        ascending=False
    ).head(20)
)

# 7. Compare Real and Fake news
for label in df["label"].dropna().unique():

    label_indices = df.index[df["label"] == label]

    label_tfidf = tfidf_matrix[label_indices].mean(axis=0).A1

    label_words = pd.DataFrame({
        "Word": tfidf_words,
        "Average_TF_IDF": label_tfidf
    })

    top_words = label_words.sort_values(
        by="Average_TF_IDF",
        ascending=False
    ).head(10)

    print(f"\nTop 10 Important Words in {label} News:")
    print(top_words)

    # Plot top words
    plt.figure(figsize=(10, 5))
    top_words.sort_values("Average_TF_IDF").plot(
        x="Word",
        y="Average_TF_IDF",
        kind="barh",
        legend=False,
        ax=plt.gca()
    )

    plt.title(f"Top 10 Important Words in {label} News")
    plt.xlabel("Average TF-IDF")
    plt.ylabel("Word")
    plt.tight_layout()
    plt.show()

# 8. Save results
tf_idf_df.to_csv("T16_tf_idf_values.csv", index=False)

print("\nTask 16 completed successfully!")
print("Results saved in T16_tf_idf_values.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'fakenews.csv'